# Machine Learning Explainability: SHAP vs LIME
### Complete Demonstration on Breast Cancer Wisconsin Dataset

**Course / Assignment:** Assignment 9: Apply SHAP/LIME for Explainability  
**Objective:** Demonstrate global and local model interpretability using Cooperative Game Theory (SHAP) and Local Surrogate Models (LIME).

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from lime.lime_tabular import LimeTabularExplainer

# Ensure project root in sys.path
sys.path.append(os.path.abspath('..'))
from src.data_loader import load_dataset, get_stratified_split
from src.train import train_model, save_model
from src.evaluate import evaluate_model
from src.shap_explainer import get_shap_explainer, compute_shap_explanation, extract_class_shap_values
from src.lime_explainer import get_lime_explainer, explain_sample_lime

## 1. Dataset Loading and Inspection

In [ ]:
X_train, X_test, y_train, y_test, meta = get_stratified_split()
print("Training samples:", X_train.shape)
print("Testing samples:", X_test.shape)
print("Class distribution:", meta['class_distribution'])
X_train.head()

## 2. Train Random Forest Model & Evaluate Performance

In [ ]:
model = train_model(X_train, y_train, n_estimators=100, random_state=42)
metrics = evaluate_model(model, X_test, y_test, output_dir="../outputs/evaluation")
print("Evaluation Metrics:")
for k, v in metrics.items():
    if k not in ['confusion_matrix', 'classification_report']:
        print(f"{k:12s}: {v:.4f}")

## 3. SHAP Explainability (Global and Local)

In [ ]:
shap_explainer = get_shap_explainer(model)
raw_exp = compute_shap_explanation(shap_explainer, X_test)
class_exp = extract_class_shap_values(raw_exp, target_class=1)

# Global Summary Beeswarm Plot
plt.figure(figsize=(10, 6))
shap.plots.beeswarm(class_exp, max_display=12)

# Local Waterfall Plot for Sample 0
plt.figure(figsize=(10, 6))
shap.plots.waterfall(class_exp[0], max_display=10)

## 4. LIME Explainability (Local Perturbation Surrogate)

In [ ]:
lime_explainer = get_lime_explainer(X_train)
exp_lime_0 = explain_sample_lime(lime_explainer, model, X_test.iloc[0], num_features=8)
fig = exp_lime_0.as_pyplot_figure()
plt.title("LIME Explanation for Sample 0 (Target: Benign)")
plt.show()

## 5. SHAP vs LIME Side-by-Side Comparison
Both explainers agree on key decisive biomarkers such as `worst concave points`, `worst perimeter`, and `worst area`.